# 🏦 StockAI — Ready-to-Use Financial AI

**No training needed. Two pre-built models, running in ~10 minutes.**

| Model | Role | HuggingFace ID |
|-------|------|----------------|
| **Finance-Chat** | Stock Q&A + Report Generation | `AdaptLLM/finance-chat` |
| **FinGPT Sentiment** | News & Tweet Sentiment Analysis | `FinGPT/fingpt-mt_llama2-7b_lora` |

---
### ⏱️ Estimated Setup Time: ~10 minutes
### ✅ Runtime Required: T4 GPU (Runtime → Change runtime type → T4)

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies (~3 min)
# ============================================================
import subprocess, sys

print('⏳ Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.41.2',
    'peft==0.11.1',
    'bitsandbytes==0.43.3',
    'accelerate==0.31.0',
    'sentencepiece',
    'scipy',
    'einops'
], check=True)

print('✅ Done. Packages ready.')

In [ ]:
# ============================================================
# CELL 2 — Load Finance-Chat Model (~5 min download)
# Core model: Stock Q&A + Full Report Generation
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

FINANCE_CHAT_ID = 'AdaptLLM/finance-chat'

print(f'📥 Loading {FINANCE_CHAT_ID}...')
print('   (First run downloads ~14GB — subsequent runs use cache)')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

finance_tokenizer = AutoTokenizer.from_pretrained(
    FINANCE_CHAT_ID,
    use_fast=False
)

finance_model = AutoModelForCausalLM.from_pretrained(
    FINANCE_CHAT_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
finance_model.eval()

print('✅ Finance-Chat model loaded and ready!')


# ── Core inference function ──
def ask_finance(question, context='', max_new_tokens=512):
    """
    Ask any financial question or request a stock report.
    
    Args:
        question: Your question or analysis request
        context:  Optional stock data (price, RSI, P/E, etc.)
        max_new_tokens: Length of response (512 = ~400 words)
    """
    if context.strip():
        prompt = f"### Instruction:\n{question}\n\n### Input:\n{context}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{question}\n\n### Response:\n"

    inputs = finance_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(finance_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = finance_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=finance_tokenizer.eos_token_id,
            pad_token_id=finance_tokenizer.eos_token_id
        )

    # Return only the generated part (not the prompt)
    generated_ids = output[0][inputs['input_ids'].shape[1]:]
    return finance_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [ ]:
# ============================================================
# CELL 3 — Load FinGPT Sentiment Model (~3 min)
# Role: Analyze news headlines and tweets for stock sentiment
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

FINGPT_BASE   = 'meta-llama/Llama-2-7b-chat-hf'   # base model
FINGPT_LORA   = 'FinGPT/fingpt-mt_llama2-7b_lora'  # LoRA adapter on top

# ── NOTE: Llama-2 requires HuggingFace token ──
# If you get an access error, run this first:
#   from huggingface_hub import login
#   login(token='hf_YOUR_TOKEN')  # https://huggingface.co/settings/tokens
#   Then accept Llama-2 license at: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
#
# ALTERNATIVE (no token needed): Use the FinBERT model below instead

USE_FINBERT_FALLBACK = True  # Set False if you have a HF token for Llama-2

if USE_FINBERT_FALLBACK:
    # ── FinBERT: Lightweight sentiment model, no token needed ──
    print('📥 Loading FinBERT (fast, no HF token needed)...')
    from transformers import pipeline

    sentiment_pipeline = pipeline(
        'text-classification',
        model='ProsusAI/finbert',
        tokenizer='ProsusAI/finbert',
        device=0 if torch.cuda.is_available() else -1
    )

    def analyze_sentiment(text):
        """
        Analyze sentiment of a news headline or tweet.
        Returns: positive / negative / neutral + confidence score
        """
        result = sentiment_pipeline(text[:512])[0]
        label = result['label'].upper()
        score = round(result['score'] * 100, 1)
        emoji = '🟢' if label == 'POSITIVE' else '🔴' if label == 'NEGATIVE' else '🟡'
        return {
            'text': text,
            'sentiment': label,
            'confidence': f'{score}%',
            'display': f"{emoji} {label} ({score}%)"
        }

    def analyze_batch(headlines: list):
        """
        Analyze multiple headlines at once.
        Returns a summary with counts and per-headline results.
        """
        results = [analyze_sentiment(h) for h in headlines]
        counts = {'POSITIVE': 0, 'NEGATIVE': 0, 'NEUTRAL': 0}
        for r in results:
            counts[r['sentiment']] += 1

        total = len(results)
        overall = max(counts, key=counts.get)
        print(f'\n📊 Sentiment Summary ({total} items):')
        print(f'   🟢 Positive: {counts["POSITIVE"]} | 🔴 Negative: {counts["NEGATIVE"]} | 🟡 Neutral: {counts["NEUTRAL"]}')
        print(f'   Overall: {overall}')
        print(f'\n📰 Per-item breakdown:')
        for r in results:
            print(f'   {r["display"]} — {r["text"][:80]}')
        return results

    print('✅ FinBERT sentiment model loaded!')

else:
    # ── Full FinGPT (requires Llama-2 HF token) ──
    print('📥 Loading FinGPT (Llama-2 base + FinGPT LoRA)...')
    sentiment_tokenizer = AutoTokenizer.from_pretrained(FINGPT_BASE)
    sentiment_base = AutoModelForCausalLM.from_pretrained(
        FINGPT_BASE, torch_dtype=torch.float16, device_map='auto'
    )
    sentiment_model = PeftModel.from_pretrained(sentiment_base, FINGPT_LORA)
    sentiment_model.eval()

    def analyze_sentiment(text):
        prompt = f"What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}\n{text}\nAnswer:"
        inputs = sentiment_tokenizer(prompt, return_tensors='pt').to(sentiment_model.device)
        with torch.no_grad():
            out = sentiment_model.generate(**inputs, max_new_tokens=10)
        answer = sentiment_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().lower()
        label = 'POSITIVE' if 'positive' in answer else 'NEGATIVE' if 'negative' in answer else 'NEUTRAL'
        emoji = '🟢' if label == 'POSITIVE' else '🔴' if label == 'NEGATIVE' else '🟡'
        return {'text': text, 'sentiment': label, 'confidence': 'N/A', 'display': f'{emoji} {label}'}

    print('✅ FinGPT sentiment model loaded!')

In [ ]:
# ============================================================
# CELL 4 — TEST: Stock Q&A
# Ask any financial question in plain English
# ============================================================

print('=' * 65)
print('TEST 1: General Financial Q&A')
print('=' * 65)

questions = [
    'What does an RSI below 30 mean and how should a trader react?',
    'Explain the difference between P/E ratio and P/B ratio.',
    'What is a golden cross pattern and is it bullish or bearish?'
]

for q in questions:
    print(f'\n❓ Q: {q}')
    print(f'💬 A: {ask_finance(q, max_new_tokens=200)}')
    print('-' * 65)

In [ ]:
# ============================================================
# CELL 5 — TEST: Full Stock Report Generation
# Paste real stock data and get a structured report
# ============================================================

print('=' * 65)
print('TEST 2: Full Investment Report — NVDA')
print('=' * 65)

# ── Paste your stock data here ──
stock_data = """
Ticker: NVDA | Company: NVIDIA Corporation | Sector: Semiconductors
Current Price: $875.40 | 52W High: $974.00 | 52W Low: $403.11
Revenue (TTM): $79.8B | EPS: $11.93 | P/E Ratio: 73.5 | P/B Ratio: 38.2
RSI (14-day): 62 | MACD: +4.2 | Trend: Bullish
Support Level: $840 | Resistance Level: $920
Debt/Equity: 0.42 | ROE: 91.4% | Institutional Holding: 68%
Chart Patterns: Cup and Handle, Golden Cross
Recent News Sentiment: Very Positive
"""

report = ask_finance(
    question='Analyze this stock and generate a complete investment report with technical analysis, fundamental analysis, and a clear BUY/SELL/HOLD recommendation with confidence level.',
    context=stock_data,
    max_new_tokens=600
)

print(report)

In [ ]:
# ============================================================
# CELL 6 — TEST: Indian Stock Analysis
# ============================================================

print('=' * 65)
print('TEST 3: Indian Stock Report — INFY (Infosys)')
print('=' * 65)

infy_data = """
Ticker: INFY | Company: Infosys Ltd | Sector: IT Services | Exchange: NSE
Current Price: ₹1,842 | 52W High: ₹2,006 | 52W Low: ₹1,358
Revenue (TTM): $18.6B USD | EPS: ₹63.4 | P/E Ratio: 29.1
RSI (14-day): 55 | MACD: +1.8 | Trend: Neutral to Bullish
Support Level: ₹1,780 | Resistance Level: ₹1,920
USD/INR Impact: Positive (weak INR boosts export revenue)
Attrition Rate: 12.7% (improving) | EBIT Margin: 21.3%
Institutional Holding: 34% FII + 18% DII = 52% total
Recent Deal Wins: $2.1B large deal TCV in Q3
News Sentiment: Positive
"""

report = ask_finance(
    question='Analyze this Indian IT stock and generate a complete investment report with buy/sell/hold signal.',
    context=infy_data,
    max_new_tokens=600
)

print(report)

In [ ]:
# ============================================================
# CELL 7 — TEST: News & Social Sentiment Engine
# Paste news headlines, tweets, CEO statements
# ============================================================

print('=' * 65)
print('TEST 4: Sentiment Analysis — NVDA News & Tweets')
print('=' * 65)

# Paste any headlines or tweets here
headlines = [
    "NVIDIA beats Q4 earnings expectations, revenue up 265% year-over-year",
    "Jensen Huang says AI demand is 'insane' and accelerating globally",
    "US government expands chip export restrictions to more countries",
    "NVDA stock falls 5% amid broader tech selloff and profit-taking",
    "Goldman Sachs raises NVDA price target to $1,100, maintains Buy rating",
    "Concerns mount over NVIDIA's high valuation at 70x forward earnings",
    "Microsoft, Google, Amazon all increasing NVIDIA GPU orders for 2025",
]

results = analyze_batch(headlines)

In [ ]:
# ============================================================
# CELL 8 — COMBINED: Full StockAI Pipeline
# News sentiment + Stock data → Final investment decision
# ============================================================

def full_stockai_analysis(ticker, company, stock_data_str, news_headlines):
    """
    Complete StockAI pipeline:
    1. Analyze news sentiment
    2. Generate full investment report
    3. Combine into final decision
    """
    print(f'\n{'='*65}')
    print(f'🏦 STOCKAI FULL ANALYSIS — {ticker} ({company})')
    print(f'{'='*65}')

    # Step 1: Sentiment
    print('\n📰 STEP 1: Analyzing News Sentiment...')
    sent_results = [analyze_sentiment(h) for h in news_headlines]
    counts = {'POSITIVE': 0, 'NEGATIVE': 0, 'NEUTRAL': 0}
    for r in sent_results:
        counts[r['sentiment']] += 1
        print(f'   {r["display"]} — {r["text"][:75]}')

    overall_sent = max(counts, key=counts.get)
    sent_score = counts['POSITIVE'] - counts['NEGATIVE']
    sentiment_summary = f"{overall_sent} (P:{counts['POSITIVE']} N:{counts['NEGATIVE']} Neu:{counts['NEUTRAL']})"
    print(f'\n   → Overall Sentiment: {sentiment_summary}')

    # Step 2: Add sentiment to stock data and generate report
    print(f'\n📊 STEP 2: Generating Investment Report...')
    enriched_data = stock_data_str + f"\nNews Sentiment Analysis: {sentiment_summary}"

    report = ask_finance(
        question=f'Generate a complete investment report for {ticker} with technical analysis, fundamental analysis, sentiment context, and a final BUY/SELL/HOLD recommendation with confidence level.',
        context=enriched_data,
        max_new_tokens=700
    )

    print(f'\n📋 FULL REPORT:')
    print(f'{'─'*65}')
    print(report)
    print(f'{'─'*65}')

    return {'ticker': ticker, 'sentiment': sentiment_summary, 'report': report}


# ── Run the full pipeline ──
result = full_stockai_analysis(
    ticker='RELIANCE',
    company='Reliance Industries',
    stock_data_str="""
    Ticker: RELIANCE | Exchange: NSE | Sector: Conglomerate
    Current Price: ₹2,945 | 52W High: ₹3,217 | 52W Low: ₹2,220
    Revenue (TTM): ₹9.74L Crore | EPS: ₹98.6 | P/E: 29.8
    RSI: 58 | MACD: +2.1 | Trend: Bullish
    Support: ₹2,840 | Resistance: ₹3,050
    Debt/Equity: 0.35 | ROE: 9.8% | FII Holding: 24%
    Key Segments: Jio (Telecom), Retail, O2C, New Energy
    """,
    news_headlines=[
        "Reliance Jio to invest $25B in 5G expansion across India",
        "Reliance Industries Q3 profit rises 9% driven by retail and digital",
        "Mukesh Ambani announces green energy target of 100GW by 2030",
        "Reliance retail faces competition from Tata and Adani groups",
        "Morgan Stanley maintains overweight rating on Reliance with ₹3,400 target",
    ]
)

In [ ]:
# ============================================================
# CELL 9 — YOUR CUSTOM ANALYSIS
# Edit the variables below to analyze any stock you want
# ============================================================

# ── EDIT THESE ──────────────────────────────────────────────
MY_TICKER   = 'AAPL'
MY_COMPANY  = 'Apple Inc'

MY_STOCK_DATA = """
Ticker: AAPL | Sector: Consumer Electronics | Exchange: NASDAQ
Current Price: $189.50
Revenue (TTM): $385.7B | EPS: $6.43 | P/E: 29.5
RSI: 52 | Trend: Neutral
Support: $182 | Resistance: $198
"""

MY_HEADLINES = [
    "Apple iPhone 16 sales disappoint in China amid Huawei competition",
    "Apple Vision Pro gaining enterprise traction, analyst says",
    "Warren Buffett reduces Berkshire's Apple stake by 13%",
]
# ── END EDIT ────────────────────────────────────────────────

result = full_stockai_analysis(MY_TICKER, MY_COMPANY, MY_STOCK_DATA, MY_HEADLINES)

---
## 🚀 How to Use in Your Platform

### Expose as a FastAPI endpoint:
```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class StockRequest(BaseModel):
    ticker: str
    stock_data: str
    headlines: list[str]

@app.post('/analyze')
def analyze(req: StockRequest):
    return full_stockai_analysis(req.ticker, '', req.stock_data, req.headlines)
```

### Deploy on HuggingFace Spaces (free hosting):
1. Create a new Space at huggingface.co/spaces
2. Choose `Gradio` as the SDK
3. Paste this into `app.py`:
```python
import gradio as gr

def gradio_analyze(ticker, stock_data, headlines_text):
    headlines = [h.strip() for h in headlines_text.split('\n') if h.strip()]
    result = full_stockai_analysis(ticker, '', stock_data, headlines)
    return result['report']

gr.Interface(
    fn=gradio_analyze,
    inputs=[
        gr.Textbox(label='Ticker (e.g. NVDA)'),
        gr.Textbox(label='Stock Data', lines=8),
        gr.Textbox(label='News Headlines (one per line)', lines=5)
    ],
    outputs=gr.Textbox(label='Investment Report', lines=20),
    title='StockAI — Financial Analyzer'
).launch()
```

---
*StockAI | AdaptLLM/finance-chat + ProsusAI/finbert*